# Module 16: ML Workflow End-to-End — Solutions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing, load_iris, load_breast_cancer
from sklearn.model_selection import (
    train_test_split, cross_val_score, learning_curve, validation_curve, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.inspection import permutation_importance
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Setup complete')

### Solution 1: Build a Complete Pipeline

In [ ]:
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=1.0, random_state=42))
])

scores = cross_val_score(pipeline, X, y, cv=5, scoring='r2')
print('=== Pipeline CV Results ===')
print(f'CV R2 scores: {np.round(scores, 4)}')
print(f'Mean R2: {scores.mean():.4f} (+/- {scores.std():.4f})')

### Solution 2: Learning Curve Analysis

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_sizes, train_scores, val_scores = learning_curve(
    pipeline, X_train, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='r2'
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, np.mean(train_scores, axis=1), 'o-', label='Training')
plt.fill_between(train_sizes,
    np.mean(train_scores, axis=1) - np.std(train_scores, axis=1),
    np.mean(train_scores, axis=1) + np.std(train_scores, axis=1), alpha=0.2)
plt.plot(train_sizes, np.mean(val_scores, axis=1), 'o-', label='Cross-validation')
plt.fill_between(train_sizes,
    np.mean(val_scores, axis=1) - np.std(val_scores, axis=1),
    np.mean(val_scores, axis=1) + np.std(val_scores, axis=1), alpha=0.2)
plt.xlabel('Training examples')
plt.ylabel('R2 score')
plt.title('Learning Curve: Ridge on California Housing')
plt.legend()
plt.grid(True)
plt.show()
print('Scores converge as training size increases. Small gap suggests low variance.')

### Solution 3: Validation Curve for alpha

In [ ]:
param_range = np.logspace(-3, 2, 10)
train_scores, val_scores = validation_curve(
    pipeline, X_train, y_train,
    param_name='regressor__alpha',
    param_range=param_range, cv=5, scoring='r2', n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.semilogx(param_range, train_mean, 'o-', label='Training')
plt.semilogx(param_range, val_mean, 'o-', label='Cross-validation')
plt.xlabel('alpha')
plt.ylabel('R2 score')
plt.title('Validation Curve: Ridge alpha')
plt.legend()
plt.grid(True)
plt.show()

best_alpha = param_range[np.argmax(val_mean)]
print(f'Optimal alpha: {best_alpha:.4f}')

### Solution 4: Confusion Matrix and Classification Report

In [ ]:
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', random_state=42))
])

svm_pipeline.fit(X_train_c, y_train_c)
y_pred_c = svm_pipeline.predict(X_test_c)

print('=== Classification Report ===')
print(classification_report(y_test_c, y_pred_c, target_names=cancer.target_names))

ConfusionMatrixDisplay.from_predictions(y_test_c, y_pred_c, display_labels=cancer.target_names)
plt.title('Breast Cancer - SVM Confusion Matrix')
plt.show()

report = classification_report(y_test_c, y_pred_c, output_dict=True)
print(f'Recall for malignant (positive): {report["malignant"]["recall"]:.4f}')

### Solution 5: Permutation Importance

In [ ]:
pipeline.fit(X_train, y_train)
perm = permutation_importance(pipeline, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)

imp_df = pd.DataFrame({
    'feature': housing.feature_names,
    'importance': perm.importances_mean,
    'std': perm.importances_std
}).sort_values('importance', ascending=False)

print('=== Top 3 Most Important Features ===')
print(imp_df.head(3).to_string(index=False))

plt.figure(figsize=(8, 4))
sns.barplot(data=imp_df, x='importance', y='feature', xerr=imp_df['std'])
plt.title('Permutation Feature Importance')
plt.tight_layout()
plt.show()

### Solution 6: Experiment Tracking with MLflow

In [ ]:
try:
    import mlflow
    from mlflow.models import infer_signature
    
    models = {
        'LinearRegression': LinearRegression(),
        'Ridge': Ridge(alpha=1.0, random_state=42),
        'RandomForest': RandomForestRegressor(n_estimators=50, random_state=42)
    }
    
    for name, model in models.items():
        with mlflow.start_run(run_name=name):
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('model', model)
            ])
            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_test)
            r2 = r2_score(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            
            mlflow.log_param('model_type', name)
            mlflow.log_metric('r2', r2)
            mlflow.log_metric('rmse', rmse)
            mlflow.sklearn.log_model(pipe, 'model', signature=infer_signature(X_train, y_train))
            print(f'{name}: R2={r2:.4f}, RMSE={rmse:.4f}')
            
    print('\nAll runs logged. Use mlflow ui to compare.')
except ImportError:
    print('MLflow not installed.')

### Solution 7: Model Serialization

In [ ]:
iris = load_iris()
X_i, y_i = iris.data, iris.target
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(X_i, y_i, test_size=0.2, random_state=42)

rf_iris = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_i, y_train_i)
y_pred_i = rf_iris.predict(X_test_i)

# Save
joblib.dump(rf_iris, 'iris_rf.joblib')
joblib.dump(rf_iris, 'iris_rf_compressed.joblib', compress=3)

# Load and verify
loaded_rf = joblib.load('iris_rf.joblib')
y_pred_loaded = loaded_rf.predict(X_test_i)

size_normal = os.path.getsize('iris_rf.joblib')
size_compressed = os.path.getsize('iris_rf_compressed.joblib')

print(f'Normal save: {size_normal / 1024:.2f} KB')
print(f'Compressed save: {size_compressed / 1024:.2f} KB')
print(f'Compression ratio: {size_normal / size_compressed:.1f}x')
print(f'Predictions match: {np.all(y_pred_i == y_pred_loaded)}')

### Solution 8: SHAP Interpretation

In [ ]:
try:
    import shap
    
    rf_cancer = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_cancer.fit(X_train_c, y_train_c)
    
    explainer = shap.TreeExplainer(rf_cancer)
    shap_values = explainer.shap_values(X_test_c[:5])
    
    # Waterfall for first prediction
    plt.figure()
    shap.waterfall_plot(shap.Explanation(
        values=shap_values[1][0], 
        base_values=explainer.expected_value[1],
        data=X_test_c[0],
        feature_names=cancer.feature_names
    ), show=False, max_display=10)
    plt.title('SHAP Waterfall Plot - First Test Sample')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print('SHAP not installed.')

### Solution 9: GridSearch with Pipeline

In [ ]:
titanic = sns.load_dataset('titanic')
titanic = titanic[['survived', 'pclass', 'sex', 'age', 'fare', 'embarked']].copy()
X_t, y_t = titanic.drop('survived', axis=1), titanic['survived']
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_t, y_t, test_size=0.2, random_state=42)

titanic_pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['age', 'fare']),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), ['pclass', 'sex', 'embarked'])
    ])),
    ('clf', RandomForestClassifier(random_state=42))
])

param_grid = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth': [3, 5, 10, None]
}

grid = GridSearchCV(titanic_pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train_t, y_train_t)

print('=== GridSearchCV Results ===')
print(f'Best params: {grid.best_params_}')
print(f'Best CV accuracy: {grid.best_score_:.4f}')
print(f'Test accuracy: {grid.score(X_test_t, y_test_t):.4f}')

### Solution 10: Complete End-to-End Workflow

In [ ]:
print('=== Complete End-to-End Workflow ===')

# 1. Pipeline with ColumnTransformer
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=42))
])

# 2. GridSearchCV
param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [5, 10, None]
}
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

print(f'Best params: {grid.best_params_}')
print(f'Best CV R2: {grid.best_score_:.4f}')
print(f'Test R2: {grid.score(X_test, y_test):.4f}')

# 3. Learning curve
best_pipeline = grid.best_estimator_
train_sizes, train_scores, val_scores = learning_curve(
    best_pipeline, X_train, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='r2'
)

plt.figure(figsize=(8, 4))
plt.plot(train_sizes, np.mean(train_scores, axis=1), 'o-', label='Training')
plt.plot(train_sizes, np.mean(val_scores, axis=1), 'o-', label='CV')
plt.xlabel('Training examples')
plt.ylabel('R2')
plt.title('Learning Curve - Best Model')
plt.legend()
plt.grid(True)
plt.show()

# 4. Permutation importance
perm = permutation_importance(best_pipeline, X_test, y_test, n_repeats=5, random_state=42)
imp_df = pd.DataFrame({'feature': housing.feature_names, 'importance': perm.importances_mean})
imp_df = imp_df.sort_values('importance', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=imp_df, x='importance', y='feature')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

# 5. MLflow logging
try:
    import mlflow
    with mlflow.start_run(run_name='e2e_california_housing'):
        mlflow.log_params(grid.best_params_)
        mlflow.log_metric('test_r2', grid.score(X_test, y_test))
        mlflow.log_metric('best_cv_r2', grid.best_score_)
        mlflow.sklearn.log_model(best_pipeline, 'model')
        plt.figure(figsize=(8, 4))
        sns.barplot(data=imp_df.head(5), x='importance', y='feature')
        plt.title('Top 5 Features')
        plt.savefig('top_features.png', bbox_inches='tight')
        mlflow.log_artifact('top_features.png')
        plt.close()
        print('Logged to MLflow')
except ImportError:
    print('MLflow not installed')

# 6. Save model
joblib.dump(best_pipeline, 'california_housing_best.joblib')
print('Model saved to california_housing_best.joblib')

# 7. Load and verify
loaded = joblib.load('california_housing_best.joblib')
y_pred_best = best_pipeline.predict(X_test[:5])
y_pred_loaded = loaded.predict(X_test[:5])
print(f'Predictions match: {np.allclose(y_pred_best, y_pred_loaded)}')

print('\n=== End-to-End Workflow Complete ===')
print('Summary of all steps executed successfully.')